# NFL Draft Prediction - Legal 10-Fold Ensemble Pipeline

This notebook uses only the competition-provided `Data/train.csv` and `Data/test.csv` files. It does not use external/source reconstruction data. The final submission is a 10-fold CatBoost ensemble trained on engineered combine features and native categorical columns.


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

TRAIN_PATH = 'Data/train.csv'
TEST_PATH = 'Data/test.csv'
OUTPUT_PATH = 'Data/submission.csv'
N_FOLDS = 10

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y = train['Drafted'].reset_index(drop=True)

cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'Target split: {int((y == 1).sum())} drafted / {int((y == 0).sum())} undrafted')


Train shape: (2781, 16)
Test shape: (696, 15)
Target split: 1803 drafted / 978 undrafted


In [6]:
def build_legal_catboost_matrix(train_df, test_df):
    df = pd.concat([train_df.drop(columns=['Drafted']), test_df], axis=0).reset_index(drop=True)
    physical_metrics = [
        'Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump',
        'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle', 'Age'
    ]

    for col in physical_metrics:
        if df[col].isnull().sum() > 0:
            df[f'{col}_Is_Missing'] = df[col].isnull().astype(int)

    for col in physical_metrics:
        df[col] = df[col].fillna(df.groupby('Position')[col].transform('median'))
        df[col] = df[col].fillna(df[col].median())

    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Speed_Weight_Ratio'] = df['Weight'] / df['Sprint_40yd']
    df['Explosiveness_Index'] = df['Vertical_Jump'] * df['Broad_Jump']
    df['Power_Score'] = df['Bench_Press_Reps'] * df['Weight']
    df['School_Combine_Count'] = df.groupby('School')['Id'].transform('count')
    df['Speed_Score'] = (df['Weight'] * 200) / (df['Sprint_40yd'] ** 4)

    for col in physical_metrics:
        df[f'{col}_Pos_Mean'] = df.groupby('Position')[col].transform('mean')
        df[f'{col}_Pos_Diff'] = df[col] - df[f'{col}_Pos_Mean']
        df.drop(columns=[f'{col}_Pos_Mean'], inplace=True)

    for col in cat_cols:
        df[col] = df[col].fillna('Unknown').astype('category')

    X_train = df.iloc[:len(train_df)].drop(columns=['Id']).reset_index(drop=True)
    X_test = df.iloc[len(train_df):].drop(columns=['Id']).reset_index(drop=True)
    return X_train, X_test

X_train, X_test = build_legal_catboost_matrix(train, test)
print(f'Feature matrix: {X_train.shape[1]} features')


Feature matrix: 36 features


In [7]:
best_catboost_params = {
    'iterations': 1183,
    'learning_rate': 0.010492928203031996,
    'depth': 4,
    'l2_leaf_reg': 4.5594175959311425,
    'min_data_in_leaf': 7,
    'verbose': 0,
    'allow_writing_files': False
}

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof = np.zeros(len(X_train))
test_preds_ensemble = np.zeros(len(X_test))
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y), start=1):
    fold_params = best_catboost_params.copy()
    fold_params['random_seed'] = 42 + fold

    model = CatBoostClassifier(**fold_params, cat_features=cat_cols)
    model.fit(
        X_train.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=(X_train.iloc[va_idx], y.iloc[va_idx]),
        early_stopping_rounds=50,
        use_best_model=True
    )

    oof[va_idx] = model.predict_proba(X_train.iloc[va_idx])[:, 1]
    test_preds_ensemble += model.predict_proba(X_test)[:, 1] / N_FOLDS

    fold_auc = roc_auc_score(y.iloc[va_idx], oof[va_idx])
    fold_scores.append(fold_auc)
    print(f'Fold {fold:02d} ROC-AUC: {fold_auc:.5f}')

print(f'Mean fold ROC-AUC: {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}')
print(f'OOF ROC-AUC: {roc_auc_score(y, oof):.5f}')


Fold 01 ROC-AUC: 0.80184
Fold 02 ROC-AUC: 0.82548
Fold 03 ROC-AUC: 0.83693
Fold 04 ROC-AUC: 0.88350
Fold 05 ROC-AUC: 0.87908
Fold 06 ROC-AUC: 0.86553
Fold 07 ROC-AUC: 0.78146
Fold 08 ROC-AUC: 0.81037
Fold 09 ROC-AUC: 0.80867
Fold 10 ROC-AUC: 0.89240
Mean fold ROC-AUC: 0.83853 +/- 0.03709
OOF ROC-AUC: 0.84074


In [ ]:
submission = test[['Id']].copy()
submission['Drafted'] = test_preds_ensemble
submission.to_csv(OUTPUT_PATH, index=False)

print(f'Saved submission: {OUTPUT_PATH}')
print(submission['Drafted'].describe())


Saved submission: Data/submission.csv
count    696.000000
mean       0.635934
std        0.287604
min        0.015050
25%        0.563083
50%        0.740950
75%        0.838635
max        0.956654
Name: Drafted, dtype: float64


: 